# Mitra Classifier — End-to-End Classification with Your Own Data

**Notebook Specification:** DIMER Notebook Specification v1.0  
**Profile:** `E2E`  
**Release status:** Candidate — static conformance checks are automated; a clean Google Colab execution of this exact revision remains the release gate.

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/mitra-classifier-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/mitra-classifier-pipeline/blob/main/tutorials/mitra_classifier_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-autogluon%2Fmitra--classifier-ffcc4d?style=flat)](https://huggingface.co/autogluon/mitra-classifier)
[![Upstream](https://img.shields.io/badge/Upstream-autogluon%2Fautogluon-181717?style=flat&logo=github&logoColor=white)](https://github.com/autogluon/autogluon)
[![arXiv](https://img.shields.io/badge/arXiv-2510.21204-b31b1b.svg)](https://arxiv.org/abs/2510.21204)

You have a labelled table (a few thousand rows, a few dozen columns, a class to predict) and you want a credible classifier today, without a hyperparameter search and without trusting a model file you cannot verify. **Mitra** is a tabular foundation model: it was pre-trained on millions of synthetic tables and predicts a new table's labels *in context*, by reading your labelled rows as its prompt. There is nothing to train for the baseline; optionally, a short fine-tune on a GPU can adapt it further.

This notebook takes you from the pinned Mitra checkpoint (from the DIMER Model Repository, or the identical upstream release) to a verified, reusable predictor ZIP, with every shortcut that quietly ruins tabular evaluations guarded against along the way.

**By the end of this notebook you will be able to:**
- **Acquire and verify** a model checkpoint by immutable revision and SHA-256, and lock the loader to that verified copy so nothing else can be substituted.
- **Prepare tabular data for honest evaluation**: keep leakage-aware splits when you have them, know when a random holdout is acceptable, and catch duplicate rows, unseen classes and too-small classes before they distort a metric.
- **Evaluate Mitra in-context**, read its metrics against the trivial baseline, and decide when fine-tuning is worth a GPU.
- **Export a predictor bundle** that reloads bit-for-bit and carries its own provenance, so a colleague can score new rows without this notebook.

**No DIMER Workbench access is required.** Data is processed in Google Colab, not by DIMER. Do not upload confidential, sensitive, or restricted data unless that environment is permitted.

**This notebook does not demonstrate:** DIMER portal serving, regression, non-tabular tasks, or production fitness. Successful tutorial execution is workflow evidence, not deployment validation.

Reference: [MODEL_CARD.md](https://github.com/kurtvalcorza/mitra-classifier-pipeline/blob/main/MODEL_CARD.md).

## Prerequisites

- **Runtime:** Google Colab with **Python 3.12**. The default pretrained/in-context path runs on CPU; a GPU is needed only for `RUN_FINE_TUNING`. Runtime varies with Colab hardware, dependency-cache state, and network throughput; this notebook does not promise a fixed completion time.
- **Knowledge:** pandas basics; what a train/validation/test split is for. You do not need to know AutoGluon; every call it makes is explained where it happens.
- **Data:** nothing, to start. The bundled FreshRetailNet sample ships with leakage-aware splits. For your own data you need a CSV with a label column and at least 50 rows, or your own `train.csv` / `val.csv` / `test.csv`.

**How to use this notebook.** Cells with a form on the right (`# @param`) are the knobs; change one and re-run from that cell down. Run everything in order the first time. Each step says what the next cell does and what to look for in its output. Steps 5 and 6 are gated on Step 4 having completed *in this session*, on purpose.


## 1. Install and inspect the runtime

This notebook installs from an embedded, exact-version lock generated from `tutorials/requirements-colab.lock.txt`. Mitra ships as an extra of AutoGluon (`autogluon.tabular[mitra]`), which pins a compatible PyTorch range and can therefore **replace Colab's preinstalled `torch`**. That is normal here, and the cell reports the actual PyTorch and CUDA versions *after* installation instead of assuming the accelerator stack was left alone. Two things to look for:

- If `torch` was already imported in this session and pip changed its version, the cell stops and asks you to *Runtime ▸ Restart session*. An imported module does not pick up a new version underneath it.
- Pip may report conflicts for unrelated preinstalled packages (`torchvision`, `diffusers`, `gradio`). This tutorial does not use them; those warnings are noise.

**What to look for:** `AutoGluon: 1.5.0` and a `CUDA available:` line. `False` is fine for the default path; fine-tuning (Step 4) will then be disabled.

**Determinism:** the notebook seeds Python, NumPy, and PyTorch where stochastic operations are used. GPU kernels and AutoGluon/PyTorch internals may still be nondeterministic, so repeated fine-tuning runs can differ slightly even with the same seed.


In [ ]:
import importlib.metadata as importlib_metadata
import sys

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(
        f'This notebook release lock targets Python 3.12, but this runtime is {sys.version.split()[0]}. '
        'Use a supported Google Colab Python 3.12 runtime, restart the session, and run top-to-bottom.'
    )

PREINSTALL_TORCH_VERSION = importlib_metadata.version('torch')
TORCH_WAS_IMPORTED = 'torch' in sys.modules
print('PyTorch before install:', PREINSTALL_TORCH_VERSION)

from pathlib import Path
LOCKED_REQUIREMENTS = '#\n# This file is autogenerated by pip-compile with Python 3.12\n# by the following command:\n#\n#    pip-compile --output-file=tutorials/requirements-colab.lock.txt --strip-extras tutorials/requirements-colab.in\n#\nantlr4-python3-runtime==4.9.3\n    # via omegaconf\nautogluon-common==1.5.0\n    # via\n    #   autogluon-core\n    #   autogluon-features\nautogluon-core==1.5.0\n    # via autogluon-tabular\nautogluon-features==1.5.0\n    # via autogluon-tabular\nautogluon-tabular==1.5.0\n    # via\n    #   -r tutorials/requirements-colab.in\n    #   autogluon-tabular\nboto3==1.43.91\n    # via\n    #   autogluon-common\n    #   autogluon-core\nbotocore==1.43.91\n    # via\n    #   boto3\n    #   s3transfer\ncertifi==2026.7.22\n    # via requests\ncharset-normalizer==3.5.1\n    # via requests\ncloudpickle==3.1.2\n    # via joblib\ncontourpy==1.3.3\n    # via matplotlib\ncycler==0.12.1\n    # via matplotlib\neinops==0.8.2\n    # via autogluon-tabular\neinx==0.4.3\n    # via autogluon-tabular\nfilelock==3.32.6\n    # via\n    #   huggingface-hub\n    #   torch\n    #   transformers\nfonttools==4.64.0\n    # via matplotlib\nfrozendict==2.4.7\n    # via einx\nfsspec==2026.7.0\n    # via\n    #   huggingface-hub\n    #   torch\nhf-xet==1.6.0\n    # via huggingface-hub\nhuggingface-hub==0.36.2\n    # via\n    #   autogluon-tabular\n    #   huggingface-hub\n    #   tokenizers\n    #   transformers\nidna==3.19\n    # via requests\njinja2==3.1.6\n    # via torch\njmespath==1.1.0\n    # via\n    #   boto3\n    #   botocore\njoblib==1.6.0\n    # via\n    #   autogluon-common\n    #   scikit-learn\nkiwisolver==1.5.1\n    # via matplotlib\nlightgbm==4.6.0\n    # via -r tutorials/requirements-colab.in\nloguru==0.7.3\n    # via autogluon-tabular\nmarkupsafe==3.0.3\n    # via jinja2\nmatplotlib==3.10.9\n    # via autogluon-core\nmpmath==1.3.0\n    # via sympy\nnetworkx==3.6.1\n    # via\n    #   autogluon-core\n    #   autogluon-tabular\n    #   torch\nnumpy==2.3.5\n    # via\n    #   autogluon-common\n    #   autogluon-core\n    #   autogluon-features\n    #   autogluon-tabular\n    #   contourpy\n    #   einx\n    #   lightgbm\n    #   matplotlib\n    #   pandas\n    #   safetensors\n    #   scikit-learn\n    #   scipy\n    #   transformers\nnvidia-cublas-cu12==12.8.4.1\n    # via\n    #   nvidia-cudnn-cu12\n    #   nvidia-cusolver-cu12\n    #   torch\nnvidia-cuda-cupti-cu12==12.8.90\n    # via torch\nnvidia-cuda-nvrtc-cu12==12.8.93\n    # via torch\nnvidia-cuda-runtime-cu12==12.8.90\n    # via torch\nnvidia-cudnn-cu12==9.10.2.21\n    # via torch\nnvidia-cufft-cu12==11.3.3.83\n    # via torch\nnvidia-cufile-cu12==1.13.1.3\n    # via torch\nnvidia-curand-cu12==10.3.9.90\n    # via torch\nnvidia-cusolver-cu12==11.7.3.90\n    # via torch\nnvidia-cusparse-cu12==12.5.8.93\n    # via\n    #   nvidia-cusolver-cu12\n    #   torch\nnvidia-cusparselt-cu12==0.7.1\n    # via torch\nnvidia-nccl-cu12==2.27.5\n    # via torch\nnvidia-nvjitlink-cu12==12.8.93\n    # via\n    #   nvidia-cufft-cu12\n    #   nvidia-cusolver-cu12\n    #   nvidia-cusparse-cu12\n    #   torch\nnvidia-nvshmem-cu12==3.3.20\n    # via torch\nnvidia-nvtx-cu12==12.8.90\n    # via torch\nomegaconf==2.3.1\n    # via autogluon-tabular\npackaging==26.3\n    # via\n    #   huggingface-hub\n    #   matplotlib\n    #   transformers\npandas==2.3.3\n    # via\n    #   autogluon-common\n    #   autogluon-core\n    #   autogluon-features\n    #   autogluon-tabular\npillow==12.3.0\n    # via matplotlib\npsutil==7.1.3\n    # via autogluon-common\npyarrow==20.0.0\n    # via autogluon-common\npyparsing==3.3.2\n    # via matplotlib\npython-dateutil==2.9.0.post0\n    # via\n    #   botocore\n    #   matplotlib\n    #   pandas\npytz==2026.3.post1\n    # via pandas\npyyaml==6.0.3\n    # via\n    #   autogluon-common\n    #   huggingface-hub\n    #   omegaconf\n    #   transformers\nregex==2026.9.10\n    # via transformers\nrequests==2.34.2\n    # via\n    #   autogluon-common\n    #   autogluon-core\n    #   huggingface-hub\n    #   transformers\ns3transfer==0.19.2\n    # via boto3\nsafetensors==0.8.0\n    # via\n    #   huggingface-hub\n    #   transformers\nscikit-learn==1.7.2\n    # via\n    #   autogluon-core\n    #   autogluon-features\n    #   autogluon-tabular\nscipy==1.16.3\n    # via\n    #   autogluon-core\n    #   autogluon-tabular\n    #   lightgbm\n    #   scikit-learn\nsix==1.17.0\n    # via python-dateutil\nsympy==1.14.0\n    # via\n    #   einx\n    #   torch\nthreadpoolctl==3.6.0\n    # via scikit-learn\ntokenizers==0.22.2\n    # via transformers\ntorch==2.9.1\n    # via\n    #   autogluon-tabular\n    #   huggingface-hub\n    #   safetensors\ntqdm==4.70.0\n    # via\n    #   autogluon-common\n    #   autogluon-core\n    #   huggingface-hub\n    #   transformers\ntransformers==4.57.6\n    # via autogluon-tabular\ntriton==3.5.1\n    # via torch\ntyping-extensions==4.16.0\n    # via\n    #   huggingface-hub\n    #   torch\ntzdata==2026.3\n    # via pandas\nurllib3==2.7.0\n    # via\n    #   botocore\n    #   requests\n\n# The following packages are considered to be unsafe in a requirements file:\n# setuptools\n'
Path('/content/mitra-requirements.lock.txt').write_text(LOCKED_REQUIREMENTS, encoding='utf-8')
%pip install -q -r /content/mitra-requirements.lock.txt

INSTALLED_TORCH_VERSION = importlib_metadata.version('torch')
AUTOGLUON_VERSION = importlib_metadata.version('autogluon.tabular')
if TORCH_WAS_IMPORTED and INSTALLED_TORCH_VERSION != PREINSTALL_TORCH_VERSION:
    raise RuntimeError('pip changed PyTorch after it had already been imported. Use Runtime → Restart session, then run the notebook top-to-bottom.')

import torch

TORCH_VERSION = torch.__version__
TORCH_CUDA_VERSION = torch.version.cuda
CUDA_AVAILABLE_AFTER_INSTALL = torch.cuda.is_available()
print('Python:', sys.version.split()[0])
print('AutoGluon:', AUTOGLUON_VERSION)
print('PyTorch after install:', TORCH_VERSION)
print('PyTorch CUDA build:', TORCH_CUDA_VERSION)
print('CUDA available:', CUDA_AVAILABLE_AFTER_INSTALL)
if not CUDA_AVAILABLE_AFTER_INSTALL:
    print('⚠ Pretrained/in-context evaluation can still run on CPU, but fine-tuning will be disabled.')

## 2. Acquire, verify, and lock the checkpoint

A model file is code you are about to run and weights you are about to trust. This step makes the supported acquisition paths explicit and checkable:

- **DIMER files** uploads exactly the two files exposed by the current DIMER model-download boundary: `model.safetensors` and `config.json`. Both are SHA-256 verified against this notebook release before either file is staged. DIMER does not currently provide this notebook with a package manifest (or equivalent producer metadata) that independently declares model identity and immutable revision, so the Notebook Spec MOD7 provenance gap remains explicit and this notebook stays **Candidate**. A failed or incomplete DIMER upload never falls back to the network.
- **Pinned upstream** fetches those same two files from the exact AutoGluon revision `c425e9fa…` on the Hugging Face Hub. A revision is an immutable commit; a model *name* is a branch that can change.

For both paths, the notebook records the intended model ID and immutable revision, verifies the exact release digests, stages the files into an isolated Hugging Face cache, and marks that cache offline. Finally it asks Hugging Face to resolve the model *exactly as AutoGluon will* and refuses to continue unless both resolved files come from that verified snapshot **and still match the expected digests**. The digest pair establishes byte identity with the pinned release; it does not manufacture DIMER-side provenance that the current producer does not emit.

If the lock check fails, use *Runtime ▸ Restart session* and run from Step 1 downward. Network requests in the pinned-upstream path use a finite timeout so outages fail clearly instead of hanging.

**What to look for:** two `✓ … verified` lines with digest prefixes `e06a055e91a3…` and `2c96c24dd25f…`, then `✓ Hugging Face resolver locked …`.

**Remote-code boundary:** this tutorial does not enable Hugging Face `trust_remote_code`; execution uses the installed AutoGluon implementation plus the verified `safetensors` weights and pinned configuration.


In [ ]:
import hashlib, json, os, random, shutil, stat, urllib.request, zipfile
from pathlib import Path, PurePosixPath

MODEL_ID = 'autogluon/mitra-classifier'
PINNED_REVISION = 'c425e9fa0910a6be1c494321792e7ba2a1367b1a'
EXPECTED_WEIGHTS_SHA256 = 'e06a055e91a3baeffc37f9cf634d9e69a27d904b6686131dc3b702f9c0126b19'
EXPECTED_CONFIG_SHA256 = '2c96c24dd25f64e92753f6f2ba00cc7833b9923459403dcd8504e8700c0995df'
NETWORK_TIMEOUT_SECONDS = 30

HF_HOME = Path('/content/mitra-hf')
MODEL_DIR = Path('/content/mitra-model')
HF_HOME.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
os.environ['HF_HOME'] = str(HF_HOME)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def fetch_pinned(name, dest):
    url = f'https://huggingface.co/{MODEL_ID}/resolve/{PINNED_REVISION}/{name}?download=true'
    print('Retrieving pinned', name)
    with urllib.request.urlopen(url, timeout=NETWORK_TIMEOUT_SECONDS) as r, open(dest, 'wb') as f:
        shutil.copyfileobj(r, f)

def verify(path, expected, label):
    actual = sha256_file(path)
    if actual != expected:
        raise RuntimeError(f'{label} checksum mismatch.\nExpected: {expected}\nActual:   {actual}')
    print(f'✓ {label} verified: {actual[:12]}…')
    return actual

def load_dimer_files(weights_dest, config_dest):
    from google.colab import files

    print('Upload exactly two files downloaded from DIMER: model.safetensors and config.json')
    uploaded = files.upload()
    expected = {
        'model.safetensors': EXPECTED_WEIGHTS_SHA256,
        'config.json': EXPECTED_CONFIG_SHA256,
    }
    if set(uploaded) != set(expected):
        raise RuntimeError(
            'DIMER files mode requires exactly model.safetensors and config.json. '
            f'Missing={sorted(set(expected) - set(uploaded))}; '
            f'unexpected={sorted(set(uploaded) - set(expected))}'
        )

    verified_payloads = {}
    for filename, expected_digest in expected.items():
        payload = uploaded[filename]
        actual_digest = hashlib.sha256(payload).hexdigest()
        if actual_digest != expected_digest:
            raise RuntimeError(
                f'DIMER {filename} checksum mismatch. '
                f'Expected {expected_digest}; got {actual_digest}.'
            )
        verified_payloads[filename] = payload
        print(f'✓ DIMER {filename} matched pinned release digest: {actual_digest[:12]}…')

    weights_dest.write_bytes(verified_payloads['model.safetensors'])
    config_dest.write_bytes(verified_payloads['config.json'])
    print(
        '✓ DIMER checkpoint pair accepted for the pinned notebook release. '
        'Producer manifest/model-revision provenance is not supplied by the current DIMER download boundary.'
    )

def install_offline_snapshot(weights, config, weights_digest):
    snapshot = weights_digest[:40]
    repo = HF_HOME / 'hub' / ('models--' + MODEL_ID.replace('/', '--'))
    snap = repo / 'snapshots' / snapshot
    refs = repo / 'refs'
    snap.mkdir(parents=True, exist_ok=True)
    refs.mkdir(parents=True, exist_ok=True)
    shutil.copy2(weights, snap / 'model.safetensors')
    shutil.copy2(config, snap / 'config.json')
    (refs / 'main').write_text(snapshot)
    os.environ['HF_HUB_OFFLINE'] = '1'
    os.environ['TRANSFORMERS_OFFLINE'] = '1'
    return snap

def assert_resolver_locked(snapshot):
    from huggingface_hub import hf_hub_download
    expected = {
        'model.safetensors': ((snapshot / 'model.safetensors').resolve(), EXPECTED_WEIGHTS_SHA256),
        'config.json': ((snapshot / 'config.json').resolve(), EXPECTED_CONFIG_SHA256),
    }
    for filename, (expected_path, expected_digest) in expected.items():
        try:
            resolved = Path(hf_hub_download(repo_id=MODEL_ID, filename=filename)).resolve()
        except Exception as exc:
            raise RuntimeError('Verified snapshot is staged but Hugging Face cannot resolve it offline. Use Runtime → Restart session and run the notebook top-to-bottom.') from exc
        if resolved != expected_path:
            raise RuntimeError(f'Offline checkpoint lock is not in effect for {filename}.\nExpected: {expected_path}\nResolved: {resolved}\nUse Runtime → Restart session and run the notebook top-to-bottom.')
        resolved_digest = sha256_file(resolved)
        if resolved_digest != expected_digest:
            raise RuntimeError(f'Resolved {filename} digest changed after staging.\nExpected: {expected_digest}\nActual:   {resolved_digest}')
    print('✓ Hugging Face resolver locked to the verified offline snapshot and digests.')

MODEL_SOURCE = 'Pinned upstream'  # @param ['DIMER files', 'Pinned upstream']

print('Model:', MODEL_ID)
print('Revision:', PINNED_REVISION)
weights_path = MODEL_DIR / 'model.safetensors'
config_path = MODEL_DIR / 'config.json'

if MODEL_SOURCE == 'DIMER files':
    load_dimer_files(weights_path, config_path)
else:
    fetch_pinned('model.safetensors', weights_path)
    fetch_pinned('config.json', config_path)

weights_digest = verify(weights_path, EXPECTED_WEIGHTS_SHA256, 'model.safetensors')
config_digest = verify(config_path, EXPECTED_CONFIG_SHA256, 'config.json')
SNAPSHOT_PATH = install_offline_snapshot(weights_path, config_path, weights_digest)
print('✓ Offline snapshot:', SNAPSHOT_PATH)
assert_resolver_locked(SNAPSHOT_PATH)


## 3. Choose a dataset

If you do not have a dataset, choose **Sample dataset (FreshRetailNet)**. The bundled ZIP contains `train.csv`, `val.csv`, and `test.csv`; the notebook preserves those partitions instead of randomly re-splitting them. It has 4,180 training rows, 1,600 validation rows, 1,600 test rows, 17 features, and a 3-class demand-band target (`low` / `mid` / `high`, roughly balanced: 31 / 35 / 33 %). It is derived from FreshRetailNet-50K, redistributed under **CC BY 4.0** for tutorial and smoke-test use, not benchmarking, and pinned to an immutable repository revision.

[Read the sample DATASET_CARD.md](https://github.com/kurtvalcorza/mitra-classifier-pipeline/blob/8fc19e80ae3166ec6bf964d194a28c80e6ba3b1f/examples/sample-data/DATASET_CARD.md)

**Why the split is preserved.** The sample is a *purged chronological split with an embargo*: training rows come before validation and test rows in time, with a gap between them. A random re-split would let the model see the future of the very series it is asked to predict, and its metrics would be flattering and useless. That is the single most common way tabular evaluations go wrong.

For your own data:

- **Upload CSV** creates a stratified random holdout and therefore assumes rows are approximately IID. Do **not** use this mode for time-dependent, grouped, panel, lagged, or rolling-window data unless random splitting is scientifically appropriate.
- **Upload pre-split train/val/test** preserves partitions you prepared externally, which is the safer choice for temporal or grouped data or any workflow with an embargo/purge rule. Feature columns may be in different orders; the notebook validates names and reorders validation/test columns to the training order.

**Checks that run on every path:** duplicate column names are rejected before pandas can rename them; rows without a label are dropped; exact duplicate rows are reported (not removed); every class needs at least 2 rows; evaluation splits may not contain classes the training split never saw; and training is capped at 10,000 rows (Mitra's supported maximum), stratified, with the cap recorded.


In [ ]:
import csv
import io
import math
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

NETWORK_TIMEOUT_SECONDS = 30

DATA_SOURCE = 'Sample dataset (FreshRetailNet)'  # @param ['Sample dataset (FreshRetailNet)', 'Upload CSV', 'Upload pre-split train/val/test']
TARGET_COLUMN = 'target'                         # @param {type:'string'}
DROP_COLUMNS = ''                                # @param {type:'string'}
VALIDATION_SPLIT = 0.20                          # @param {type:'number'}
SEED = 42                                        # @param {type:'integer'}

SAMPLE_REVISION = '8fc19e80ae3166ec6bf964d194a28c80e6ba3b1f'
SAMPLE_ZIP_URL = f'https://raw.githubusercontent.com/kurtvalcorza/mitra-classifier-pipeline/{SAMPLE_REVISION}/examples/sample-data/freshretailnet-band-h7.zip'
SAMPLE_CARD_URL = f'https://github.com/kurtvalcorza/mitra-classifier-pipeline/blob/{SAMPLE_REVISION}/examples/sample-data/DATASET_CARD.md'

using_sample = DATA_SOURCE == 'Sample dataset (FreshRetailNet)'
using_presplit_data = DATA_SOURCE in {'Sample dataset (FreshRetailNet)', 'Upload pre-split train/val/test'}
test_data = None
TRAIN_ROW_CAP_APPLIED = False

def uploaded_csv(payload, label='uploaded CSV'):
    # Check original names before pandas can rename duplicate headers.
    text = payload.decode('utf-8-sig')
    rows = csv.reader(io.StringIO(text, newline=''))
    header = next((row for row in rows if row and not (len(row) == 1 and not row[0].strip())), [])
    seen = set()
    duplicates = []
    for name in header:
        if name in seen and name not in duplicates:
            duplicates.append(name)
        seen.add(name)
    if duplicates:
        raise ValueError(f'{label} contains duplicate column names: {duplicates}')
    return pd.read_csv(io.BytesIO(payload))

def read_presplit_upload():
    from google.colab import files
    uploaded = files.upload()
    by_base = {Path(name).name.lower(): payload for name, payload in uploaded.items()}
    required = {'train.csv', 'val.csv', 'test.csv'}
    missing = sorted(required - set(by_base))
    if missing:
        raise RuntimeError(f'Upload train.csv, val.csv, and test.csv together. Missing: {missing}')
    return (
        uploaded_csv(by_base['train.csv'], 'train.csv'),
        uploaded_csv(by_base['val.csv'], 'val.csv'),
        uploaded_csv(by_base['test.csv'], 'test.csv'),
    )

if using_sample:
    with urllib.request.urlopen(SAMPLE_ZIP_URL, timeout=NETWORK_TIMEOUT_SECONDS) as r:
        payload = r.read()
    with zipfile.ZipFile(io.BytesIO(payload)) as z:
        names = {Path(n).name: n for n in z.namelist() if not n.endswith('/')}
        required = {'train.csv', 'val.csv', 'test.csv'}
        missing = sorted(required - set(names))
        if missing:
            raise RuntimeError(f'Sample ZIP missing: {missing}')
        train_data = uploaded_csv(z.read(names['train.csv']), 'train.csv')
        holdout_data = uploaded_csv(z.read(names['val.csv']), 'val.csv')
        test_data = uploaded_csv(z.read(names['test.csv']), 'test.csv')
    TARGET_COLUMN = 'target'
    print('✓ Using FreshRetailNet sample with preserved train/val/test splits.')
    print('  Sample revision:', SAMPLE_REVISION)
    print('  Dataset card:', SAMPLE_CARD_URL)
elif DATA_SOURCE == 'Upload pre-split train/val/test':
    train_data, holdout_data, test_data = read_presplit_upload()
    print('✓ Using uploaded train/val/test partitions without re-splitting.')
else:
    from google.colab import files
    uploaded = files.upload()
    csvs = [(name, payload) for name, payload in uploaded.items() if name.lower().endswith('.csv')]
    if len(csvs) != 1:
        raise RuntimeError('Upload exactly one labelled CSV.')
    data = uploaded_csv(csvs[0][1])
    print('⚠ Upload CSV uses a stratified random holdout and assumes rows are IID. For temporal/grouped/lagged data, use the pre-split option.')

drop_columns = [c.strip() for c in DROP_COLUMNS.split(',') if c.strip() and c.strip() != TARGET_COLUMN]

def prepare(df, name):
    if df.columns.duplicated().any():
        raise ValueError(f'{name}: duplicate column names are not supported.')
    if TARGET_COLUMN not in df.columns:
        raise ValueError(f'{name}: target {TARGET_COLUMN!r} not found.')
    dropped_feature_columns = [c for c in drop_columns if c in df.columns]
    null_target_rows = int(df[TARGET_COLUMN].isna().sum())
    out = df.drop(columns=dropped_feature_columns, errors='ignore').dropna(subset=[TARGET_COLUMN]).copy()
    if dropped_feature_columns or null_target_rows:
        print(f'ℹ {name}: preprocessing modified the input: dropped feature columns={dropped_feature_columns or []}; rows dropped for null target={null_target_rows}.')
    duplicate_rows = int(out.duplicated().sum())
    if duplicate_rows:
        print(f'⚠ {name}: {duplicate_rows:,} exact duplicate labelled rows detected; inspect for leakage or accidental copies.')
    features = [c for c in out.columns if c != TARGET_COLUMN]
    counts = out[TARGET_COLUMN].value_counts()
    errors = []
    if len(out) < 50:
        errors.append('use at least 50 labelled rows')
    if not features:
        errors.append('no feature columns remain')
    if len(features) > 500:
        errors.append(f'{len(features)} features exceed the 500-feature limit')
    if not 2 <= len(counts) <= 10:
        errors.append(f'target has {len(counts)} classes; Mitra requires 2–10')
    if counts.empty or counts.min() < 2:
        errors.append('every class needs at least 2 rows')
    if errors:
        raise ValueError(f'{name} is not ready: ' + '; '.join(errors))
    return out, features

def require_class_coverage(train_frame, eval_frame, eval_name):
    train_classes = set(train_frame[TARGET_COLUMN].dropna().unique())
    eval_classes = set(eval_frame[TARGET_COLUMN].dropna().unique())
    missing = sorted(train_classes - eval_classes, key=str)
    unseen = sorted(eval_classes - train_classes, key=str)
    problems = []
    if missing:
        problems.append(f'missing trained target classes: {missing}')
    if unseen:
        problems.append(f'contains unseen target classes not present in training: {unseen}')
    if problems:
        raise ValueError(f'{eval_name} ' + '; '.join(problems) + '. Adjust the split or provide a compatible evaluation partition.')

if using_presplit_data:
    train_data, features = prepare(train_data, 'train.csv')
    holdout_data, val_features = prepare(holdout_data, 'val.csv')
    test_data, test_features = prepare(test_data, 'test.csv')
    train_feature_set = set(features)
    if set(val_features) != train_feature_set or set(test_features) != train_feature_set:
        raise ValueError('train/val/test feature column names do not match.')
    ordered_columns = features + [TARGET_COLUMN]
    holdout_data = holdout_data.reindex(columns=ordered_columns)
    test_data = test_data.reindex(columns=ordered_columns)
    require_class_coverage(train_data, holdout_data, 'validation split')
    require_class_coverage(train_data, test_data, 'test split')
else:
    clean, features = prepare(data, 'uploaded CSV')
    if not 0.05 <= VALIDATION_SPLIT <= 0.40:
        raise ValueError('VALIDATION_SPLIT must be 0.05–0.40.')
    n_classes = clean[TARGET_COLUMN].nunique()
    n_holdout = math.ceil(len(clean) * VALIDATION_SPLIT)
    n_train = len(clean) - n_holdout
    if n_holdout < n_classes or n_train < n_classes:
        raise ValueError(f'VALIDATION_SPLIT yields train={n_train}, holdout={n_holdout} for {n_classes} classes. Both partitions need at least one row per class; increase rows or adjust VALIDATION_SPLIT.')
    train_data, holdout_data = train_test_split(
        clean,
        test_size=VALIDATION_SPLIT,
        random_state=SEED,
        stratify=clean[TARGET_COLUMN],
    )
    require_class_coverage(train_data, holdout_data, 'holdout')

def require_min_training_class_count(frame, context):
    counts = frame[TARGET_COLUMN].value_counts()
    too_small = counts[counts < 2]
    if not too_small.empty:
        raise ValueError(
            f'{context} leaves fewer than 2 training rows for class(es) {too_small.to_dict()}. '
            'Use the pre-split upload path, add more rows for rare classes, or adjust VALIDATION_SPLIT.'
        )

require_min_training_class_count(train_data, 'Training split')

TRAIN_ROWS_BEFORE_CAP = len(train_data)
if len(train_data) > 10_000:
    try:
        train_data, _ = train_test_split(
            train_data,
            train_size=10_000,
            random_state=SEED,
            stratify=train_data[TARGET_COLUMN],
        )
    except ValueError as exc:
        raise ValueError(
            'Could not cap the training split to 10,000 rows while preserving class representation. '
            'Use pre-split train/val/test files or increase support for rare classes.'
        ) from exc
    TRAIN_ROW_CAP_APPLIED = True
    require_min_training_class_count(train_data, 'Capped training split')

FEATURE_COLUMNS = [c for c in train_data.columns if c != TARGET_COLUMN]
NUM_CLASSES = train_data[TARGET_COLUMN].nunique()
PROBLEM_TYPE = 'binary' if NUM_CLASSES == 2 else 'multiclass'

display(pd.DataFrame({
    'Item': ['Training rows', 'Holdout rows', 'Independent test rows', 'Features', 'Target', 'Classes'],
    'Value': [len(train_data), len(holdout_data), len(test_data) if test_data is not None else None, len(FEATURE_COLUMNS), TARGET_COLUMN, NUM_CLASSES],
}))
display(pd.concat([
    train_data[TARGET_COLUMN].value_counts().rename('training'),
    holdout_data[TARGET_COLUMN].value_counts().rename('holdout'),
], axis=1).fillna(0).astype(int))
if test_data is not None:
    display(test_data[TARGET_COLUMN].value_counts().rename('independent test').to_frame())
if TRAIN_ROW_CAP_APPLIED:
    print(f'⚠ Training rows capped from {TRAIN_ROWS_BEFORE_CAP:,} to {len(train_data):,}.')
if len(train_data) > 5_000:
    print("⚠ Above Mitra's particularly strong reported ≤5,000-sample regime.")
if len(FEATURE_COLUMNS) > 100:
    print("⚠ Above Mitra's particularly strong reported ≤100-feature regime.")


## 4. Evaluate pretrained Mitra, then optionally fine-tune

**What "pretrained evaluation" means here.** With `fine_tune=False`, Mitra does not update a single weight. AutoGluon hands it your training rows as *context*, and for each validation row the model predicts a label by attending over that context. `predictor.fit(...)` still runs, because that is AutoGluon's API, but what it does is register the context and pick the model configuration; the 300 MB of weights are the same bytes you verified in Step 2.

When a pre-split source is used, `val.csv` is the **holdout** (used for reporting and, if fine-tuning runs, for choosing between pretrained and fine-tuned) and `test.csv` is the **independent test**, reported but never used for any decision. That separation is what keeps the test numbers honest.

**Fine-tuning** (`RUN_FINE_TUNING`, GPU only) trains the weights for `FINE_TUNE_STEPS` steps on your training rows. The default path leaves it off, so by default this step shows a *before* without an *after*; switch it on to get the comparison table.

**Colab memory note:** `MAX_MEMORY_USAGE_RATIO=1.10` is the setting that cleared AutoGluon's default 0.90 safety gate on a standard Tesla T4 Colab in the development run. AutoGluon may still print a warning suggesting a larger value (that run suggested `>=1.27`). Do **not** raise the ratio merely to silence the warning: values above 1.0 intentionally accept more out-of-memory risk. If fitting is skipped or fails, first reduce rows or context, or use a higher-memory runtime.

**Fine-tuning note:** `FINE_TUNE_STEPS=50` makes the requested schedule explicit. `FINE_TUNE_TIME_LIMIT` can truncate that schedule, so a time-limited run should not be interpreted as a controlled 50-step experiment.

**Metric direction:** `log_loss` is lower-is-better; the other displayed metrics are higher-is-better. AutoGluon returns lower-is-better metrics with their sign flipped; the helper converts `log_loss` back to a conventional positive loss before display and export.

**Rerun safety:** this step clears predictor objects and output directories from any earlier execution before fitting, so a failed rerun cannot leave an older model eligible for inference or export. If AutoGluon still reports insufficient RAM in a fresh run, restart the session or use a higher-memory runtime rather than raising the safety ratio.

**Selection guard:** when fine-tuning ran, the notebook recommends pretrained or fine-tuned by `EVAL_METRIC` on the holdout only, and only if the holdout has at least `MIN_SELECTION_HOLDOUT_ROWS = 50` rows; below that it keeps the pretrained predictor and reports the fine-tuned metrics as evidence. A worse independent-test metric after fine-tuning is printed as a warning and never changes the selection.

**Reproducibility note:** `SEED` fixes the notebook-controlled random choices, but exact fine-tuning reproducibility is not guaranteed across GPU kernels, driver/runtime revisions, or AutoGluon/PyTorch internals. Treat small run-to-run deltas accordingly.


In [ ]:
import gc
import shutil
from pathlib import Path

import torch
from autogluon.tabular import TabularPredictor

EVAL_METRIC = 'accuracy'           # @param ['accuracy', 'balanced_accuracy', 'log_loss', 'f1_macro', 'mcc']
BASELINE_TIME_LIMIT = 300          # @param {type:'integer'}
RUN_FINE_TUNING = False            # @param {type:'boolean'}
FINE_TUNE_STEPS = 50               # @param {type:'integer'}
FINE_TUNE_TIME_LIMIT = 600         # @param {type:'integer'}
MAX_MEMORY_USAGE_RATIO = 1.10      # @param {type:'number'}
MIN_SELECTION_HOLDOUT_ROWS = 50

BASELINE_PATH = Path('/content/mitra-baseline')
FINETUNED_PATH = Path('/content/mitra-finetuned')
EXPORT_ZIP_PATH = Path('/content/mitra-predictor.zip')

# A failed rerun must never leave an earlier predictor eligible for inference/export.
FIT_RUN_COMPLETED = False
for stale_name in (
    'active_predictor',
    'recommended_predictor',
    'active_mode',
    'selection_basis',
    'baseline_predictor',
    'finetuned_predictor',
    'baseline_metrics',
    'baseline_test_metrics',
    'finetuned_metrics',
    'finetuned_test_metrics',
):
    globals().pop(stale_name, None)

for stale_path in (BASELINE_PATH, FINETUNED_PATH):
    shutil.rmtree(stale_path, ignore_errors=True)
EXPORT_ZIP_PATH.unlink(missing_ok=True)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

baseline_predictor = None
finetuned_predictor = None
baseline_metrics = None
baseline_test_metrics = None
finetuned_metrics = None
finetuned_test_metrics = None

CUDA_AVAILABLE = torch.cuda.is_available()
print('PyTorch:', torch.__version__)
print('PyTorch CUDA build:', torch.version.cuda)
print('CUDA available:', CUDA_AVAILABLE, torch.cuda.get_device_name(0) if CUDA_AVAILABLE else '')
print(f'AutoGluon memory safety ratio: {MAX_MEMORY_USAGE_RATIO:.2f}')
print('✓ Cleared stale predictor state and output paths before fitting.')

def seed_everything():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

def fit_mitra(fine_tune, path, time_limit, steps=None):
    seed_everything()
    hp = {'fine_tune': fine_tune, 'seed': SEED}
    if EVAL_METRIC in {'accuracy', 'log_loss'}:
        hp['metric'] = EVAL_METRIC
    if fine_tune:
        if steps is None or steps <= 0:
            raise ValueError('FINE_TUNE_STEPS must be a positive integer when fine-tuning is enabled.')
        hp['fine_tune_steps'] = steps
    predictor = TabularPredictor(
        label=TARGET_COLUMN,
        problem_type=PROBLEM_TYPE,
        eval_metric=EVAL_METRIC,
        path=str(path),
        verbosity=2,
    )
    predictor.fit(
        train_data,
        hyperparameters={'MITRA': hp},
        fit_weighted_ensemble=False,
        time_limit=time_limit,
        ag_args_fit={'max_memory_usage_ratio': MAX_MEMORY_USAGE_RATIO},
    )
    if not any('mitra' in n.lower() for n in predictor.model_names()):
        raise RuntimeError(f'Expected Mitra; AutoGluon trained {predictor.model_names()}.')
    return predictor

def metrics(predictor, frame):
    raw = predictor.evaluate(frame, auxiliary_metrics=True, silent=True)
    return {k: float(-v if k == 'log_loss' else v) for k, v in raw.items()}

def metric_table(values, name):
    frame = pd.DataFrame({'value': pd.Series(values)})
    frame['direction'] = ['lower is better' if metric == 'log_loss' else 'higher is better' for metric in frame.index]
    frame.columns = [name, 'direction']
    return frame

baseline_predictor = fit_mitra(False, BASELINE_PATH, BASELINE_TIME_LIMIT)
baseline_metrics = metrics(baseline_predictor, holdout_data)
baseline_test_metrics = metrics(baseline_predictor, test_data) if test_data is not None else None
display(metric_table(baseline_metrics, 'Pretrained — holdout'))
if baseline_test_metrics is not None:
    display(metric_table(baseline_test_metrics, 'Pretrained — independent test'))

if RUN_FINE_TUNING:
    if not CUDA_AVAILABLE:
        raise RuntimeError('Fine-tuning requires a GPU. Choose Runtime → Change runtime type → GPU.')
    finetuned_predictor = fit_mitra(
        True,
        FINETUNED_PATH,
        FINE_TUNE_TIME_LIMIT,
        FINE_TUNE_STEPS,
    )
    finetuned_metrics = metrics(finetuned_predictor, holdout_data)
    comparison = pd.DataFrame({'Pretrained': baseline_metrics, 'Fine-tuned': finetuned_metrics})
    comparison['direction'] = ['lower is better' if metric == 'log_loss' else 'higher is better' for metric in comparison.index]
    display(comparison)
    one_row_accuracy = 1 / len(holdout_data)
    print(f'Holdout size: {len(holdout_data):,}; one-row accuracy resolution: {one_row_accuracy:.6f}.')
    print('Interpret deltas smaller than a few holdout rows cautiously. The time limit may also truncate the requested fine-tune schedule.')
    if test_data is not None:
        finetuned_test_metrics = metrics(finetuned_predictor, test_data)
        display(metric_table(finetuned_test_metrics, 'Fine-tuned — independent test'))
else:
    print('Fine-tuning skipped. Set RUN_FINE_TUNING=True on a GPU to run it.')


def metric_is_better(candidate, baseline, metric_name):
    if metric_name not in candidate or metric_name not in baseline:
        raise RuntimeError(f'Metric {metric_name!r} was not returned by AutoGluon; cannot select a predictor safely.')
    if metric_name == 'log_loss':
        return candidate[metric_name] < baseline[metric_name]
    return candidate[metric_name] > baseline[metric_name]


def metric_is_worse(candidate, baseline, metric_name):
    if metric_name not in candidate or metric_name not in baseline:
        return False
    if metric_name == 'log_loss':
        return candidate[metric_name] > baseline[metric_name]
    return candidate[metric_name] < baseline[metric_name]


recommended_predictor = baseline_predictor
active_mode = 'pretrained'
selection_basis = 'default:pretrained'
if finetuned_predictor is not None:
    if len(holdout_data) < MIN_SELECTION_HOLDOUT_ROWS:
        selection_basis = (
            f'default:pretrained; holdout-too-small:'
            f'{len(holdout_data)}<{MIN_SELECTION_HOLDOUT_ROWS}'
        )
        print(
            '⚠ Holdout is too small for automatic model selection '
            f'({len(holdout_data)} rows; minimum {MIN_SELECTION_HOLDOUT_ROWS}). '
            'Keeping the pretrained predictor for inference/export. '
            'Fine-tuned metrics are still reported as evaluation evidence.'
        )
    else:
        selection_basis = f'holdout:{EVAL_METRIC}'
        if metric_is_better(finetuned_metrics, baseline_metrics, EVAL_METRIC):
            recommended_predictor = finetuned_predictor
            active_mode = 'fine-tuned'
        print(
            f'✓ Recommended predictor for inference/export: {active_mode} '
            f'(selected by {EVAL_METRIC} on the holdout: '
            f'pretrained={baseline_metrics[EVAL_METRIC]:.6g}, fine-tuned={finetuned_metrics[EVAL_METRIC]:.6g}).'
        )
    if finetuned_test_metrics is not None and baseline_test_metrics is not None:
        degraded = [
            metric_name for metric_name in baseline_test_metrics
            if metric_name in finetuned_test_metrics
            and metric_is_worse(finetuned_test_metrics, baseline_test_metrics, metric_name)
        ]
        if degraded:
            details = ', '.join(
                f'{name}: {baseline_test_metrics[name]:.6g} → {finetuned_test_metrics[name]:.6g}'
                for name in degraded
            )
            print(
                '⚠ Fine-tuning produced mixed independent-test evidence. '
                f'These metrics worsened: {details}. Selection never uses the independent test; '
                'it remains evaluation evidence only.'
            )
else:
    print('✓ Recommended predictor for inference/export: pretrained (fine-tuning not run).')

active_predictor = recommended_predictor

FIT_RUN_COMPLETED = True
print('✓ Step 4 completed successfully; this run is eligible for inference/export.')


## 4b. Companion classical tree baselines & in-memory post-hoc ensembling

Evaluate lightweight classical tree baselines (**LightGBM** and **Random Forest**) on the exact same (capped) training rows (`train_data`) and holdout partition (`holdout_data`) to benchmark Mitra's foundation model performance and demonstrate in-memory probability ensembling with strict label alignment.
**Baseline variability:** LightGBM and Random Forest both use `random_state=SEED` and `n_estimators=100`; the blend searches a fixed 101-point weight grid. The seeded tree baselines are intended to be repeatable under the locked software stack, while Mitra GPU execution may retain framework/kernel nondeterminism described in Step 1.


In [ ]:
import time
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, log_loss, roc_auc_score
from sklearn.preprocessing import OrdinalEncoder

if not globals().get('FIT_RUN_COMPLETED', False) or baseline_predictor is None:
    raise RuntimeError('Run Step 4 successfully before running baseline comparisons.')

print('--- Step 4b: Classical Tree Baselines & In-Memory Ensembling (Holdout) ---')

# 1. Fit classical tree baselines with train-fitted categorical preprocessing for BYOD robustness
cat_cols = [c for c in FEATURE_COLUMNS if not pd.api.types.is_numeric_dtype(train_data[c])]
num_cols = [c for c in FEATURE_COLUMNS if pd.api.types.is_numeric_dtype(train_data[c])]

# keep_empty_features=True ensures all-NaN numeric columns in BYOD CSVs are imputed to 0.0 rather than dropped
num_imputer = SimpleImputer(strategy='median', keep_empty_features=True) if num_cols else None
cat_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1) if cat_cols else None


def fit_transform_trees(train_df):
    parts = []
    if num_cols:
        arr_num = num_imputer.fit_transform(train_df[num_cols])
        parts.append(pd.DataFrame(arr_num, columns=num_cols, index=train_df.index))
    if cat_cols:
        # Note: .astype(str) maps missing categoricals to 'nan' as an observed category
        cat_str = train_df[cat_cols].astype(str)
        arr_cat = cat_encoder.fit_transform(cat_str)
        parts.append(pd.DataFrame(arr_cat, columns=cat_cols, index=train_df.index))
    return pd.concat(parts, axis=1)[FEATURE_COLUMNS]


def transform_trees(eval_df):
    parts = []
    if num_cols:
        arr_num = num_imputer.transform(eval_df[num_cols])
        parts.append(pd.DataFrame(arr_num, columns=num_cols, index=eval_df.index))
    if cat_cols:
        cat_str = eval_df[cat_cols].astype(str)
        arr_cat = cat_encoder.transform(cat_str)
        parts.append(pd.DataFrame(arr_cat, columns=cat_cols, index=eval_df.index))
    return pd.concat(parts, axis=1)[FEATURE_COLUMNS]


X_tr_tree = fit_transform_trees(train_data)
y_tr = train_data[TARGET_COLUMN]

t0_lgbm = time.perf_counter()
lgbm_model = LGBMClassifier(
    random_state=SEED,
    n_estimators=100,
    verbose=-1,
)
lgbm_model.fit(X_tr_tree, y_tr)
t_fit_lgbm = time.perf_counter() - t0_lgbm

t0_rf = time.perf_counter()
rf_model = RandomForestClassifier(
    random_state=SEED,
    n_estimators=100,
)
rf_model.fit(X_tr_tree, y_tr)
t_fit_rf = time.perf_counter() - t0_rf

# 2. Strict class & probability alignment check
class_labels = list(active_predictor.class_labels)
missing_lgbm = set(class_labels) - set(lgbm_model.classes_)
if missing_lgbm:
    raise RuntimeError(f'LightGBM never saw classes: {sorted(missing_lgbm)}')
missing_rf = set(class_labels) - set(rf_model.classes_)
if missing_rf:
    raise RuntimeError(f'Random Forest never saw classes: {sorted(missing_rf)}')

lgbm_class_to_idx = {cls: idx for idx, cls in enumerate(lgbm_model.classes_)}
rf_class_to_idx = {cls: idx for idx, cls in enumerate(rf_model.classes_)}
reorder_lgbm = [lgbm_class_to_idx[cls] for cls in class_labels]
reorder_rf = [rf_class_to_idx[cls] for cls in class_labels]


def get_tree_proba(model, X, reorder):
    raw_fn = getattr(model, 'predict_proba')
    raw = np.asarray(raw_fn(X), dtype=float)
    return raw[:, reorder]


def timed_eval(model_type, model, frame, dev_name, reorder=None, repeats=5):
    if model_type == 'mitra':
        X = frame[FEATURE_COLUMNS]
        if repeats > 1:
            # Discarded warm-up call to eliminate first-call setup/autotuning
            _ = model.predict_proba(X, as_multiclass=True)
        latencies = []
        for _ in range(repeats):
            t0 = time.perf_counter()
            prob_df = model.predict_proba(X, as_multiclass=True)
            latencies.append((time.perf_counter() - t0) * 1000.0)
        prob = prob_df[class_labels].to_numpy(dtype=float)
    else:
        if repeats > 1:
            # Discarded warm-up call
            _ = get_tree_proba(model, transform_trees(frame), reorder)
        latencies = []
        for _ in range(repeats):
            t0 = time.perf_counter()
            X_tree = transform_trees(frame)
            prob = get_tree_proba(model, X_tree, reorder)
            latencies.append((time.perf_counter() - t0) * 1000.0)
    lat_ms = float(np.median(latencies))
    pred = [class_labels[i] for i in np.argmax(prob, axis=1)]
    return pred, prob, lat_ms, dev_name


dev_mitra = 'cuda' if torch.cuda.is_available() else 'cpu'
pred_mitra_h, prob_mitra_h, lat_mitra_h, _ = timed_eval('mitra', active_predictor, holdout_data, dev_mitra)
pred_lgbm_h, prob_lgbm_h, lat_lgbm_h, _ = timed_eval('tree', lgbm_model, holdout_data, 'cpu', reorder_lgbm)
pred_rf_h, prob_rf_h, lat_rf_h, _ = timed_eval('tree', rf_model, holdout_data, 'cpu', reorder_rf)

y_h = holdout_data[TARGET_COLUMN].tolist()
n_h = len(holdout_data)
one_row_h_pct = (1.0 / n_h) * 100.0 if n_h > 0 else 0.0


def score_classification(y_true, y_pred, y_prob):
    acc = float(accuracy_score(y_true, y_pred))
    bal_acc = float(balanced_accuracy_score(y_true, y_pred))
    loss = float(log_loss(y_true, y_prob, labels=class_labels))
    try:
        if len(class_labels) == 2:
            auc = float(roc_auc_score(y_true, y_prob[:, 1]))
        else:
            auc = float(roc_auc_score(y_true, y_prob, multi_class='ovr', labels=class_labels))
    except Exception:
        auc = float('nan')
    return {'accuracy': acc, 'balanced_acc': bal_acc, 'roc_auc': auc, 'log_loss': loss}


m_mitra_h = score_classification(y_h, pred_mitra_h, prob_mitra_h)
m_lgbm_h = score_classification(y_h, pred_lgbm_h, prob_lgbm_h)
m_rf_h = score_classification(y_h, pred_rf_h, prob_rf_h)

print(f'\nHoldout sample count: {n_h:,} rows (training cap: 10,000; 1 row = {one_row_h_pct:.4f}% of holdout)')
print(f"{'Model':<20} | {'Device':<6} | {'Rows':<6} | {'Accuracy':<10} | {'ROC-AUC':<10} | {'Log Loss':<10} | {'Latency':<10}")
print('-' * 89)
print(f"{'Mitra (' + active_mode + ')':<20} | {dev_mitra:<6} | {n_h:<6} | {m_mitra_h['accuracy']:<10.4f} | {m_mitra_h['roc_auc']:<10.4f} | {m_mitra_h['log_loss']:<10.4f} | {lat_mitra_h:<8.2f} ms")
print(f"{'LightGBM':<20} | {'cpu':<6} | {n_h:<6} | {m_lgbm_h['accuracy']:<10.4f} | {m_lgbm_h['roc_auc']:<10.4f} | {m_lgbm_h['log_loss']:<10.4f} | {lat_lgbm_h:<8.2f} ms")
print(f"{'Random Forest':<20} | {'cpu':<6} | {n_h:<6} | {m_rf_h['accuracy']:<10.4f} | {m_rf_h['roc_auc']:<10.4f} | {m_rf_h['log_loss']:<10.4f} | {lat_rf_h:<8.2f} ms")

# 3. In-memory post-hoc convex probability blend (optimizing holdout accuracy with plateau tie-breaking)
best_acc = -1.0
tied_weights = []
grid = np.linspace(0.0, 1.0, 101)
for w in grid:
    blend_prob = w * prob_mitra_h + (1.0 - w) * prob_lgbm_h
    blend_pred = [class_labels[i] for i in np.argmax(blend_prob, axis=1)]
    acc = float(accuracy_score(y_h, blend_pred))
    if acc > best_acc + 1e-9:
        best_acc = acc
        tied_weights = [float(w)]
    elif abs(acc - best_acc) <= 1e-9:
        tied_weights.append(float(w))

# Select an actual member from tied_weights (upper-median element, which breaks even ties toward the foundation model)
best_w = tied_weights[len(tied_weights) // 2]
blend_prob_h = best_w * prob_mitra_h + (1.0 - best_w) * prob_lgbm_h
blend_pred_h = [class_labels[i] for i in np.argmax(blend_prob_h, axis=1)]
m_blend_h = score_classification(y_h, blend_pred_h, blend_prob_h)
assert abs(m_blend_h["accuracy"] - best_acc) <= 1e-9, f"Selected weight {best_w} accuracy ({m_blend_h['accuracy']:.4f}) does not match best ({best_acc:.4f})"
acc_delta_h = m_blend_h['accuracy'] - m_mitra_h['accuracy']

print(f'\n[Post-Hoc Ensembling] Declared objective: maximize holdout accuracy')
print(f'Optimal holdout blend weight: {best_w:.2f} Mitra + {1.0 - best_w:.2f} LightGBM ({len(tied_weights)} weights tied at max accuracy {best_acc:.4f})')
print(f"Holdout Blend: Accuracy={m_blend_h['accuracy']:.4f} (delta vs Mitra: {acc_delta_h:+.4f}), ROC-AUC={m_blend_h['roc_auc']:.4f}, Log Loss={m_blend_h['log_loss']:.4f}")

# 4. Evaluate generalization on independent test set if present
if test_data is not None:
    pred_mitra_t, prob_mitra_t, _, _ = timed_eval('mitra', active_predictor, test_data, dev_mitra, repeats=1)
    pred_lgbm_t, prob_lgbm_t, _, _ = timed_eval('tree', lgbm_model, test_data, 'cpu', reorder=reorder_lgbm, repeats=1)
    pred_rf_t, prob_rf_t, _, _ = timed_eval('tree', rf_model, test_data, 'cpu', reorder=reorder_rf, repeats=1)
    blend_prob_t = best_w * prob_mitra_t + (1.0 - best_w) * prob_lgbm_t
    blend_pred_t = [class_labels[i] for i in np.argmax(blend_prob_t, axis=1)]
    y_t = test_data[TARGET_COLUMN].tolist()
    n_t = len(test_data)
    one_row_t_pct = (1.0 / n_t) * 100.0 if n_t > 0 else 0.0

    m_mitra_t = score_classification(y_t, pred_mitra_t, prob_mitra_t)
    m_lgbm_t = score_classification(y_t, pred_lgbm_t, prob_lgbm_t)
    m_rf_t = score_classification(y_t, pred_rf_t, prob_rf_t)
    m_blend_t = score_classification(y_t, blend_pred_t, blend_prob_t)
    acc_delta_t = m_blend_t['accuracy'] - m_mitra_t['accuracy']

    print(f'\n[Independent Test Generalization] {n_t:,} rows (1 row = {one_row_t_pct:.4f}% of test):')
    print(f"  Mitra Test: Accuracy={m_mitra_t['accuracy']:.4f}, ROC-AUC={m_mitra_t['roc_auc']:.4f}, Log Loss={m_mitra_t['log_loss']:.4f}")
    print(f"  LightGBM Test: Accuracy={m_lgbm_t['accuracy']:.4f}, ROC-AUC={m_lgbm_t['roc_auc']:.4f}, Log Loss={m_lgbm_t['log_loss']:.4f}")
    print(f"  Random Forest Test: Accuracy={m_rf_t['accuracy']:.4f}, ROC-AUC={m_rf_t['roc_auc']:.4f}, Log Loss={m_rf_t['log_loss']:.4f}")
    print(f"  Ensemble Test: Accuracy={m_blend_t['accuracy']:.4f} (delta vs Mitra: {acc_delta_t:+.4f}), ROC-AUC={m_blend_t['roc_auc']:.4f}, Log Loss={m_blend_t['log_loss']:.4f}")

    if acc_delta_h > 0 and acc_delta_t <= 0:
        print('⚠ Mixed-evidence warning: Holdout blend improved accuracy, but independent test accuracy degraded.')
        print('  Holdout weight selection may be slightly overfitted; treat holdout gain as selection-biased.')
else:
    print('ℹ No independent test partition present; holdout ensemble score is selection-biased demonstration evidence.')

# 5. In-memory fast CPU latency comparison
print(f'\n[In-Memory Latency Summary] Batch of {n_h:,} rows (median of 5 warmed runs):')
print(f'  Mitra ({dev_mitra}): {lat_mitra_h:.2f} ms')
print(f'  LightGBM (cpu): {lat_lgbm_h:.2f} ms ({lat_mitra_h / max(lat_lgbm_h, 0.01):.1f}x speedup on CPU)')
print(f'  Random Forest (cpu): {lat_rf_h:.2f} ms ({lat_mitra_h / max(lat_rf_h, 0.01):.1f}x speedup on CPU)')
print('Note: the blend is evaluated in memory only; Step 6 exports the unchanged model bundle.')

**What the baseline and ensemble numbers show.**
- **Label & probability alignment:** LightGBM probability columns are explicitly mapped to match `active_predictor.class_labels` prior to convex combination, preventing silent label permutation errors.
- **Single-objective blending:** The convex blend optimizes strictly for holdout accuracy. Reporting all metrics alongside the one-row resolution ($1/N$) ensures small metric differences are interpreted transparently.
- **Export contract unchanged:** The post-hoc blend is evaluated strictly in-memory. The exported artifact in Step 6 remains the pure, validated AutoGluon model bundle (`mitra-predictor.zip`).

**What the numbers mean.** On the bundled sample the pretrained model scores about **0.58 accuracy on the holdout and 0.55 on the independent test** (development runs; yours will be close but not identical). The classes are nearly balanced, so always predicting the biggest class would score **0.33**: Mitra is doing real work with no training at all, and the gap between holdout and test is the kind of drift a chronological split is supposed to reveal.

- `balanced_accuracy` near `accuracy` confirms no single class is carrying the score.
- `log_loss` around 0.9 on three classes means the probabilities are informative but far from confident (a perfect model scores 0, a uniform guess scores 1.10). Use it alongside accuracy when probability quality matters; calibration itself is not established by this notebook and must be evaluated separately for the deployment domain.
- With fine-tuning on, the comparison table shows both runs side by side. In the development run, 50 fine-tuning steps moved the holdout from 0.579 to 0.586 (ten rows out of 1,600) while the independent test went from 0.551 to 0.546, so the notebook recommended the fine-tuned predictor on the holdout and printed the mixed-evidence warning. That is exactly the pattern that should make you distrust a small holdout win. The printed *one-row accuracy resolution* (1 / holdout rows) is the smallest change the holdout can even measure; a delta smaller than a few rows' worth is noise.
- These are **optimization and evaluation** numbers for this sample. Whether 0.58 is *good* depends on what a wrong demand band costs in your business; that judgement is yours, not the notebook's.

**Try it:** switch `EVAL_METRIC` to `log_loss` and re-run this step; the recommended predictor may change, because accuracy and probability quality do not always agree.


## 5. Classify new rows

Upload an unlabelled CSV with the same feature columns (order does not matter; extra columns are kept in the output but not used). For a pre-split dataset, `test.csv` was already scored above; this step is for genuinely new rows. It is off by default (`RUN_NEW_DATA_INFERENCE`) so that a top-to-bottom run needs no upload dialog.

The output adds a `prediction` column and one `probability_<class>` column per class. The cell refuses to run unless Step 4 completed in this session, which is why the flag `FIT_RUN_COMPLETED` exists.

**Decision rule and uncertainty:** `predict()` returns the class selected by the predictor; for standard multiclass use this is equivalent to choosing the class with the highest returned class probability. The probability vector is useful for ranking and thresholding, but calibration for your deployment population has not been established by this notebook.


In [ ]:
import csv
import io


def read_inference_csv(payload):
    # Check original names before pandas can rename duplicate headers.
    text = payload.decode('utf-8-sig')
    rows = csv.reader(io.StringIO(text, newline=''))
    header = next((row for row in rows if row and not (len(row) == 1 and not row[0].strip())), [])
    seen = set()
    duplicates = []
    for name in header:
        if name in seen and name not in duplicates:
            duplicates.append(name)
        seen.add(name)
    if duplicates:
        raise ValueError(f'Inference CSV contains duplicate column names: {duplicates}')
    return pd.read_csv(io.BytesIO(payload))

import pandas as pd

RUN_NEW_DATA_INFERENCE = False  # @param {type:'boolean'}

if RUN_NEW_DATA_INFERENCE:
    if not globals().get('FIT_RUN_COMPLETED', False) or baseline_predictor is None:
        raise RuntimeError('No predictor was successfully trained in this Step 4 execution. Run Step 4 successfully before inference.')
    from google.colab import files
    uploaded = files.upload()
    csvs = [(name, payload) for name, payload in uploaded.items() if name.lower().endswith('.csv')]
    if len(csvs) != 1:
        raise RuntimeError('Upload exactly one inference CSV.')
    new_data = read_inference_csv(csvs[0][1])
    if new_data.columns.duplicated().any():
        duplicates = list(new_data.columns[new_data.columns.duplicated()])
        raise ValueError(f'Inference CSV contains duplicate column names: {duplicates}')
    missing = [c for c in FEATURE_COLUMNS if c not in new_data.columns]
    if missing:
        raise ValueError(f'Inference CSV is missing required features: {missing}')
    X = new_data.reindex(columns=FEATURE_COLUMNS).copy()
    active = active_predictor
    pred = active.predict(X)
    proba = active.predict_proba(X, as_multiclass=True)
    reserved_output_columns = ['prediction'] + [f'probability_{label}' for label in proba.columns]
    output_collisions = [name for name in reserved_output_columns if name in new_data.columns]
    if output_collisions:
        raise ValueError(
            f'Inference CSV contains output column(s) reserved by this notebook: {output_collisions}. '
            'Rename or remove them before inference.'
        )
    out = new_data.copy()
    out['prediction'] = pred.values
    for col in proba.columns:
        out[f'probability_{col}'] = proba[col].values
    out.to_csv('/content/predictions.csv', index=False)
    display(out.head())
    files.download('/content/predictions.csv')
else:
    print('Inference skipped.')


## 6. Export the reusable predictor

The AutoGluon `TabularPredictor` directory is the reusable trained artifact: it holds the Mitra weights *and* the training context the model needs at prediction time, plus AutoGluon's preprocessing state. The exported ZIP is therefore not a replacement `model.safetensors`; extract it and load the directory with `TabularPredictor.load(path)`. Reload it with the same runtime recorded in `tutorial_run_metadata.json` (this tutorial pins `autogluon.tabular[mitra]==1.5.0`).

`tutorial_run_metadata.json` is written into the predictor directory so the bundle explains itself: checkpoint identity and digests, runtime versions, feature list, data source and row counts, the metric used, whether fine-tuning ran, the selection basis, and every metric table from Step 4.

The export cell refuses to package a predictor unless the **current Step 4 execution** completed successfully, so an older in-memory predictor or a leftover directory cannot be mistaken for the result of a failed rerun.

**Trust boundary, for whoever receives the ZIP:** `TabularPredictor.load` deserializes Python objects (pickle). The companion inference notebook checks the archive's SHA-256 when you give it one, and refuses unsafe archive paths, but nothing makes an untrusted predictor archive safe to load. Print and keep the ZIP digest.

**Data disclosure warning:** this predictor bundle contains the fitted preprocessing state and Mitra's training/support context. Depending on the data and AutoGluon serialization details, the archive may retain values derived from or copied from the training dataset. Apply the same confidentiality, licensing, retention, and sharing rules to the predictor ZIP that apply to the source data.

The export also writes `artifact-manifest.json` with an explicit artifact format/version plus per-file sizes and SHA-256 digests, then prints the SHA-256 of the final ZIP for transfer verification.


In [ ]:
import importlib.metadata as importlib_metadata
import json
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path

if not globals().get('FIT_RUN_COMPLETED', False) or baseline_predictor is None:
    raise RuntimeError(
        'No predictor was successfully trained in this Step 4 execution. '
        'Run Step 4 successfully before exporting.'
    )

active_predictor = recommended_predictor
active_path = Path(active_predictor.path)
if not active_path.exists() or not any('mitra' in name.lower() for name in active_predictor.model_names()):
    raise RuntimeError('The current predictor artifact is missing or does not contain Mitra; refusing to export.')
metadata = {
    'artifact_format': 'dimer-mitra-autogluon-predictor',
    'artifact_format_version': '1.0',
    'base_model': MODEL_ID,
    'base_model_revision': PINNED_REVISION,
    'weights_sha256': EXPECTED_WEIGHTS_SHA256,
    'config_sha256': EXPECTED_CONFIG_SHA256,
    'model_source': MODEL_SOURCE,
    'autogluon_version': importlib_metadata.version('autogluon.tabular'),
    'torch_version': torch.__version__,
    'torch_cuda_version': torch.version.cuda,
    'python_version': sys.version.split()[0],
    'cuda_available': torch.cuda.is_available(),
    'determinism_note': 'Python/NumPy/PyTorch are seeded; GPU kernels and framework internals may remain nondeterministic.',
    'exported_at_utc': datetime.now(timezone.utc).isoformat(),
    'mode': active_mode,
    'selection_basis': selection_basis,
    'target_column': TARGET_COLUMN,
    'features': FEATURE_COLUMNS,
    'seed': SEED,
    'data_source': DATA_SOURCE,
    'sample_revision': SAMPLE_REVISION if using_sample else None,
    'sample_dataset_card': SAMPLE_CARD_URL if using_sample else None,
    'train_rows_before_cap': TRAIN_ROWS_BEFORE_CAP,
    'train_rows_used': len(train_data),
    'train_row_cap_applied': TRAIN_ROW_CAP_APPLIED,
    'holdout_rows': len(holdout_data),
    'independent_test_rows': len(test_data) if test_data is not None else None,
    'eval_metric': EVAL_METRIC,
    'baseline_time_limit_seconds': BASELINE_TIME_LIMIT,
    'fine_tuning_requested': RUN_FINE_TUNING,
    'fine_tune_steps_requested': FINE_TUNE_STEPS if RUN_FINE_TUNING else None,
    'fine_tune_time_limit_seconds': FINE_TUNE_TIME_LIMIT if RUN_FINE_TUNING else None,
    'fine_tune_schedule_may_be_truncated_by_time_limit': bool(RUN_FINE_TUNING),
    'max_memory_usage_ratio': MAX_MEMORY_USAGE_RATIO,
    'holdout_metrics_pretrained': baseline_metrics,
    'holdout_metrics_finetuned': finetuned_metrics,
    'independent_test_metrics_pretrained': baseline_test_metrics,
    'independent_test_metrics_finetuned': finetuned_test_metrics,
    'ai_assistance': {
        'client': 'GPT-5.6 Sol High',
        'agent_relay_role': 'Builder',
        'note': 'Attribution is provenance, not sign-off or independent verification.',
    },
}
(active_path / 'tutorial_run_metadata.json').write_text(json.dumps(metadata, indent=2))

manifest_files = []
for file_path in sorted(p for p in active_path.rglob('*') if p.is_file() and p.name != 'artifact-manifest.json'):
    manifest_files.append({
        'path': file_path.relative_to(active_path).as_posix(),
        'size_bytes': file_path.stat().st_size,
        'sha256': sha256_file(file_path),
    })
artifact_manifest = {
    'artifact_format': 'dimer-mitra-autogluon-predictor',
    'artifact_format_version': '1.0',
    'base_model': MODEL_ID,
    'base_model_revision': PINNED_REVISION,
    'files': manifest_files,
}
(active_path / 'artifact-manifest.json').write_text(json.dumps(artifact_manifest, indent=2))

Path('/content/mitra-predictor.zip').unlink(missing_ok=True)
archive = shutil.make_archive('/content/mitra-predictor', 'zip', root_dir=active_path)
archive_sha256 = sha256_file(archive)
print('✓ Predictor archive:', archive)
print('SHA-256:', archive_sha256)


## 7. Reload smoke test

Before treating the ZIP as reusable, this cell extracts the **exported archive** into a fresh directory, applies the same archive-safety, exact-manifest, provenance, runtime, and offline-loading gates used by the companion notebook, then reloads it with `TabularPredictor.load(...)` and confirms that predictions and probabilities match the in-memory predictor on a small holdout sample. Class predictions must be identical; probabilities are compared with `np.allclose` at a relative tolerance of 1e-6, which is the level at which floating-point summation order, not the model, explains a difference.

The reload fails before deserialization if required manifest files, digests, model identity, revision, format version, feature schema, or runtime provenance are absent or inconsistent. This is the actual packaging boundary downstream users rely on.


In [ ]:
import json
import os
import shutil
import stat
import zipfile
from pathlib import Path, PurePosixPath

from autogluon.tabular import TabularPredictor

RELOAD_DIR = Path('/content/mitra-predictor-reload')
MAX_RELOAD_EXPANDED_BYTES = 4 * 1024 ** 3
if RELOAD_DIR.exists():
    shutil.rmtree(RELOAD_DIR)
RELOAD_DIR.mkdir(parents=True)


def safe_extract_predictor_archive(zip_path, destination):
    destination = destination.resolve()
    seen = set()
    expanded_bytes = 0
    with zipfile.ZipFile(zip_path) as z:
        for info in z.infolist():
            if '\\' in info.filename:
                raise RuntimeError(f'Backslash archive member paths are not allowed: {info.filename!r}')
            member = PurePosixPath(info.filename)
            if member.is_absolute() or '..' in member.parts:
                raise RuntimeError(f'Unsafe archive member path: {info.filename!r}')
            mode = (info.external_attr >> 16) & 0o170000
            if mode == stat.S_IFLNK:
                raise RuntimeError(f'Symlink entries are not allowed: {info.filename!r}')
            normalized = member.as_posix()
            if normalized in seen:
                raise RuntimeError(f'Duplicate archive member path: {normalized!r}')
            seen.add(normalized)
            expanded_bytes += int(info.file_size)
            if expanded_bytes > MAX_RELOAD_EXPANDED_BYTES:
                raise RuntimeError('Predictor archive exceeds the 4 GiB expanded-size safety limit.')
            target = (destination / Path(*member.parts)).resolve()
            if target != destination and destination not in target.parents:
                raise RuntimeError(f'Archive member escapes extraction root: {info.filename!r}')
        z.extractall(destination)


safe_extract_predictor_archive(archive, RELOAD_DIR)
reload_candidates = sorted({p.parent.resolve() for p in RELOAD_DIR.rglob('predictor.pkl')})
if len(reload_candidates) != 1:
    raise RuntimeError(
        f'Expected exactly one AutoGluon predictor root after reload extraction; found {len(reload_candidates)}.'
    )
reload_predictor_root = reload_candidates[0]

all_extracted_files = [p.resolve() for p in RELOAD_DIR.rglob('*') if p.is_file()]
outside_predictor_root = [
    p for p in all_extracted_files
    if p != reload_predictor_root and reload_predictor_root not in p.parents
]
if outside_predictor_root:
    raise RuntimeError(
        'Reload archive contains file(s) outside the predictor root: '
        f'{[str(p.relative_to(RELOAD_DIR.resolve())) for p in outside_predictor_root]}'
    )

reload_manifest_path = reload_predictor_root / 'artifact-manifest.json'
reload_metadata_path = reload_predictor_root / 'tutorial_run_metadata.json'
if not reload_manifest_path.exists():
    raise RuntimeError('Reload artifact is missing required artifact-manifest.json.')
if not reload_metadata_path.exists():
    raise RuntimeError('Reload artifact is missing required tutorial_run_metadata.json.')

reload_manifest = json.loads(reload_manifest_path.read_text())
if reload_manifest.get('artifact_format') != 'dimer-mitra-autogluon-predictor' or reload_manifest.get('artifact_format_version') != '1.0':
    raise RuntimeError('Reload artifact manifest format/version is unsupported.')
if reload_manifest.get('base_model') != MODEL_ID or reload_manifest.get('base_model_revision') != PINNED_REVISION:
    raise RuntimeError('Reload artifact manifest model identity/revision is inconsistent.')
reload_entries = reload_manifest.get('files')
if not isinstance(reload_entries, list) or not reload_entries:
    raise RuntimeError('Reload artifact manifest must contain a non-empty files list.')

reload_listed = {}
for entry in reload_entries:
    if not isinstance(entry, dict) or not {'path', 'size_bytes', 'sha256'} <= set(entry):
        raise RuntimeError(f'Malformed reload artifact manifest entry: {entry!r}')
    rel = str(entry['path'])
    rel_path = PurePosixPath(rel)
    if '\\' in rel or rel_path.is_absolute() or '..' in rel_path.parts or rel in reload_listed:
        raise RuntimeError(f'Unsafe or duplicate reload artifact manifest path: {rel!r}')
    if rel == 'artifact-manifest.json':
        raise RuntimeError('artifact-manifest.json must not list itself.')
    reload_listed[rel] = entry

reload_actual = {
    p.relative_to(reload_predictor_root).as_posix()
    for p in reload_predictor_root.rglob('*')
    if p.is_file() and p.resolve() != reload_manifest_path.resolve()
}
reload_expected = set(reload_listed)
if reload_actual != reload_expected:
    raise RuntimeError(
        f'Reload artifact manifest file set mismatch. Missing={sorted(reload_expected - reload_actual)}; '
        f'unexpected={sorted(reload_actual - reload_expected)}'
    )
for rel, entry in reload_listed.items():
    p = reload_predictor_root / Path(*PurePosixPath(rel).parts)
    if p.stat().st_size != int(entry['size_bytes']):
        raise RuntimeError(f'Reload artifact manifest size mismatch for {rel!r}.')
    expected_digest = str(entry['sha256']).lower()
    if len(expected_digest) != 64 or any(ch not in '0123456789abcdef' for ch in expected_digest):
        raise RuntimeError(f'Reload artifact manifest has invalid SHA-256 for {rel!r}.')
    if sha256_file(p) != expected_digest:
        raise RuntimeError(f'Reload artifact manifest SHA-256 mismatch for {rel!r}.')
print(f'✓ Reload artifact manifest verified: {len(reload_listed)} files, exact file set, sizes, and SHA-256 digests.')

reload_metadata = json.loads(reload_metadata_path.read_text())
required_reload_provenance = [
    'artifact_format', 'artifact_format_version', 'base_model', 'base_model_revision',
    'autogluon_version', 'weights_sha256', 'config_sha256', 'features',
]
missing_reload_provenance = [k for k in required_reload_provenance if not reload_metadata.get(k)]
if missing_reload_provenance:
    raise RuntimeError(f'Reload artifact provenance is incomplete: {missing_reload_provenance}')

reload_feature_columns = reload_metadata['features']
if (
    not isinstance(reload_feature_columns, list)
    or not reload_feature_columns
    or any(not isinstance(name, str) or not name.strip() for name in reload_feature_columns)
    or len(set(reload_feature_columns)) != len(reload_feature_columns)
):
    raise RuntimeError(
        'Reload artifact provenance features must be a non-empty list of unique, non-blank string column names.'
    )
if reload_feature_columns != list(FEATURE_COLUMNS):
    raise RuntimeError(
        'Reload artifact provenance feature schema does not match the producing notebook feature schema.'
    )
print('✓ Reload feature schema validated before deserialization.')
if reload_metadata['artifact_format'] != 'dimer-mitra-autogluon-predictor' or reload_metadata['artifact_format_version'] != '1.0':
    raise RuntimeError('Reload artifact provenance format/version is unsupported.')
if reload_metadata['base_model'] != MODEL_ID or reload_metadata['base_model_revision'] != PINNED_REVISION:
    raise RuntimeError('Reload artifact provenance model identity/revision is inconsistent.')
if reload_metadata['weights_sha256'] != EXPECTED_WEIGHTS_SHA256 or reload_metadata['config_sha256'] != EXPECTED_CONFIG_SHA256:
    raise RuntimeError('Reload artifact provenance weight/config digests are inconsistent.')
if reload_metadata['autogluon_version'] != AUTOGLUON_VERSION:
    raise RuntimeError(
        f"Reload artifact expects AutoGluon {reload_metadata['autogluon_version']}, but runtime has {AUTOGLUON_VERSION}."
    )
if reload_manifest.get('base_model') != reload_metadata['base_model'] or reload_manifest.get('base_model_revision') != reload_metadata['base_model_revision']:
    raise RuntimeError('Reload artifact manifest and provenance disagree on model identity/revision.')

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
print('✓ Reload provenance validated; network/model fallback disabled before deserialization.')

reloaded_predictor = TabularPredictor.load(str(reload_predictor_root))
reloaded_feature_metadata = getattr(reloaded_predictor, 'feature_metadata_in', None)
if reloaded_feature_metadata is None:
    raise RuntimeError('Reloaded predictor does not expose feature_metadata_in; cannot reconcile artifact feature schema.')
reloaded_predictor_features = list(reloaded_feature_metadata.get_features())
if reloaded_predictor_features != reload_feature_columns:
    raise RuntimeError(
        'Reload artifact feature schema disagrees with the loaded predictor feature schema.'
    )
print('✓ Reload artifact feature schema reconciled with loaded predictor.')
smoke_X = holdout_data[FEATURE_COLUMNS].head(5).copy()

expected_pred = active_predictor.predict(smoke_X).reset_index(drop=True)
reloaded_pred = reloaded_predictor.predict(smoke_X).reset_index(drop=True)
if not expected_pred.equals(reloaded_pred):
    raise RuntimeError('Reload smoke test failed: class predictions changed after ZIP export/reload.')

expected_proba = active_predictor.predict_proba(smoke_X, as_multiclass=True).reset_index(drop=True)
reloaded_proba = reloaded_predictor.predict_proba(smoke_X, as_multiclass=True).reset_index(drop=True)
if list(expected_proba.columns) != list(reloaded_proba.columns):
    raise RuntimeError('Reload smoke test failed: probability class columns changed after reload.')
if not np.allclose(expected_proba.to_numpy(), reloaded_proba.to_numpy(), rtol=1e-6, atol=1e-8):
    raise RuntimeError('Reload smoke test failed: class probabilities changed after ZIP export/reload.')

print('✓ Exported predictor passed the downstream artifact boundary and reproduced smoke-test predictions.')


## What a successful run proves, and what to do next

If every cell ran in a clean runtime, this session has shown: the checkpoint you evaluated is the pinned release, byte for byte; your data passed the leakage and coverage checks; Mitra's in-context accuracy on your holdout (and, if you switched it on, after fine-tuning); and a predictor ZIP that reloads and reproduces its predictions. It has **not** shown that the model is fit for your decision. That needs your own held-out data from the period or population you will deploy on, a cost-aware metric, and, for consequential uses, subgroup and drift checks.

### Recap against the objectives
- *Acquire and verify:* Step 2 (revision + SHA-256 + resolver lock).
- *Prepare data for honest evaluation:* Step 3 (preserved splits, duplicate and class-coverage checks, row cap).
- *Evaluate in context and read the numbers:* Step 4 and the notes after it.
- *Export a self-describing bundle:* Steps 6–7.

### Next experiments, in the order they teach the most
1. Set `RUN_FINE_TUNING = True` on a GPU runtime and compare the two rows of the table; then set `FINE_TUNE_STEPS = 200` and see whether the holdout keeps improving while the independent test does not (that gap is overfitting to the holdout).
2. Change `EVAL_METRIC` to `log_loss`; note whether the recommended predictor changes.
3. Upload your own single CSV, then the same data as pre-split files with a time-based split, and compare the two accuracies. The difference is the leakage the random split hides.
4. Feed the exported ZIP to the companion [inference notebook](mitra_classifier_predictor_inference_colab.ipynb) with a fresh CSV.



### Companion predictor inference notebook

Once you have exported `mitra-predictor.zip` from Step 6, use the companion inference notebook to reload the standalone predictor, validate a new unlabelled CSV, and download `predictions.csv` without re-running training or fine-tuning:

[![Open Predictor Inference In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/mitra-classifier-pipeline/blob/main/tutorials/mitra_classifier_predictor_inference_colab.ipynb)

### Troubleshooting
| Symptom | Cause | What to do |
|---|---|---|
| `checksum mismatch` in Step 2 | an uploaded DIMER file or upstream download is not the pinned release | re-download; never edit the expected digest |
| `Offline checkpoint lock is not in effect` | `huggingface_hub` was imported before the cache was configured (cells run out of order) | *Runtime ▸ Restart session*, run from Step 1 |
| `… is not ready: target has N classes` / `every class needs at least 2 rows` | Mitra supports 2–10 classes and needs ≥2 rows per class | merge rare classes or add rows |
| `contains unseen target classes` | a validation/test class never appears in training | fix the split; the model cannot predict a class it never saw |
| AutoGluon skips Mitra for memory / `Expected Mitra; AutoGluon trained […]` | the memory guard refused the fit | fewer rows or features, a higher-memory runtime; raise `MAX_MEMORY_USAGE_RATIO` only as a last resort |
| `No predictor was successfully trained in this Step 4 execution` | Step 5/6 run before or after a failed Step 4 | re-run Step 4 successfully first |
| `Predictor was exported with AutoGluon X, but this runtime has Y` (inference notebook) | version drift | install the recorded version |

### Licences and provenance
The sample is CC BY 4.0 (FreshRetailNet-50K derivative); Mitra is Apache-2.0 from the AutoGluon team at AWS; DIMER redistributes the pinned `model.safetensors` and is not the model developer. All of it is recorded in `tutorial_run_metadata.json` inside the bundle.


## AI use and provenance

This tutorial was developed with substantial AI assistance under human direction and review.

- Original build: **GPT-5.6 Sol High** (OpenAI / ChatGPT), Agent Relay role: **Builder**
- Content revision (structure, explanations, troubleshooting): **Claude Fable 5.1** (Anthropic / Claude Code), Agent Relay role: **Reviewer and Builder**
- Review refinement & baseline expansion: **Gemini 3.8 Flash High** (Google DeepMind / Antigravity), Agent Relay role: **Builder**
- Base-model developer: **AutoGluon team, Amazon Web Services (AWS)**
- DIMER role: distributor of the pinned `model.safetensors` artifact, not model developer

AI attribution is **provenance, not sign-off** and does not independently verify correctness. Executed checks and reproducible outputs remain the evidence for a particular run.
